In [1]:
# Importando a biblioteca Pandas (O "Excel" do Python)
import pandas as pd

# Carregando a base de dados oficial do Santander
df = pd.read_csv('train.csv')

print("Base de dados carregada com sucesso!")

Base de dados carregada com sucesso!


In [2]:
# Quantas linhas (clientes) e colunas (variáveis) nós temos?
print(f"Linhas e Colunas: {df.shape}\n")

# Mostrando as 5 primeiras linhas da tabela
display(df.head())

Linhas e Colunas: (76020, 371)



,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.170000,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.030000,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.770000,0
3,8,2,37,0.0,195.0,195.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64007.970000,0
4,10,2,39,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,117310.979016,0


In [4]:
# Contagem absoluta de clientes satisfeitos (0) e insatisfeitos (1)
contagem_absoluta = df['TARGET'].value_counts()

# Transformando em porcentagem (%)
percentual = df['TARGET'].value_counts(normalize=True) * 100

print("--- QUANTIDADE DE CLIENTES ---")
print(contagem_absoluta)
print("\n--- TAXA DE INSATISFAÇÃO (%) ---")
print(percentual.round(2)) # O round(2) arredonda para duas casas decimais

--- QUANTIDADE DE CLIENTES ---
TARGET
0    73012
1     3008
Name: count, dtype: int64

--- TAXA DE INSATISFAÇÃO (%) ---
TARGET
0    96.04
1     3.96
Name: proportion, dtype: float64


In [3]:
# Resumo técnico da base: tipos de dados e contagem de valores não-nulos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


In [5]:
# 1. Procurando colunas onde só existe 1 único valor igual para todo mundo
colunas_constantes = [col for col in df.columns if df[col].nunique() == 1]

# 2. Mostrando quantas são
print(f"Colunas 100% inúteis (constantes) encontradas: {len(colunas_constantes)}")

# 3. Jogando essas colunas no lixo (removendo do DataFrame)
df = df.drop(columns=colunas_constantes)

# 4. Conferindo o novo tamanho da nossa base
print(f"Novo tamanho da base após a faxina: {df.shape}")

Colunas 100% inúteis (constantes) encontradas: 34
Novo tamanho da base após a faxina: (76020, 337)


In [6]:
# 1. Renomeando a coluna 'var15' para 'idade' para facilitar a nossa análise
df = df.rename(columns={'var15': 'idade'})

# 2. Agrupando os clientes por satisfação (0 ou 1) e tirando a média e resumo da idade
resumo_idade = df.groupby('TARGET')['idade'].describe()

print("--- RESUMO DA IDADE POR PERFIL DE SATISFAÇÃO ---")
print(resumo_idade.round(1))

--- RESUMO DA IDADE POR PERFIL DE SATISFAÇÃO ---
          count  mean   std   min   25%   50%   75%    max
TARGET                                                    
0       73012.0  32.9  12.9   5.0  23.0  27.0  39.0  105.0
1        3008.0  39.7  12.0  23.0  30.0  38.0  47.0  102.0


In [7]:
# 1. Calculando a correlação de todas as variáveis APENAS com a coluna TARGET
# Isso é muito mais rápido do que cruzar a base inteira
correlacoes = df.corrwith(df['TARGET']).sort_values()

# 2. Mostrando os extremos: O que segura o cliente e o que expulsa o cliente
print("--- TOP 5 PRODUTOS/FATORES QUE MAIS SEGURAM O CLIENTE NO BANCO ---")
print(correlacoes.head(5))

print("\n--- TOP 5 FATORES QUE MAIS INDICAM QUE O CLIENTE VAI CANCELAR ---")
# Usamos tail(6)[:-1] para pegar os maiores, ignorando a última linha que é o próprio TARGET (que daria 100%)
print(correlacoes.tail(6)[:-1])

--- TOP 5 PRODUTOS/FATORES QUE MAIS SEGURAM O CLIENTE NO BANCO ---
ind_var30             -0.149811
num_meses_var5_ult3   -0.148253
num_var30             -0.138289
num_var42             -0.135693
ind_var5              -0.135349
dtype: float64

--- TOP 5 FATORES QUE MAIS INDICAM QUE O CLIENTE VAI CANCELAR ---
imp_op_var39_efect_ult1    0.030380
num_var8_0                 0.046622
ind_var8_0                 0.046665
idade                      0.101322
var36                      0.102919
dtype: float64


In [8]:
# 1. Separando apenas o nosso Fato (TARGET) e as Dimensões que descobrimos que importam
colunas_ouro = [
    'TARGET', 'idade', 'var36', 'ind_var8_0', 'num_var8_0', 'imp_op_var39_efect_ult1',
    'ind_var30', 'num_meses_var5_ult3', 'num_var30', 'num_var42', 'ind_var5'
]

# 2. Criando um novo DataFrame leve e focado no negócio
df_powerbi = df[colunas_ouro]

# 3. Exportando o arquivo limpo para a sua pasta
df_powerbi.to_csv('santander_churn_powerbi.csv', index=False)

print("✅ Arquivo 'santander_churn_powerbi.csv' gerado com sucesso!")
print(f"O tamanho da nossa base caiu de 337 colunas para apenas {df_powerbi.shape[1]} colunas de puro valor.")

✅ Arquivo 'santander_churn_powerbi.csv' gerado com sucesso!
O tamanho da nossa base caiu de 337 colunas para apenas 11 colunas de puro valor.
